# Named Entity Recognition (NER) - Case Study
## Information Extraction from News Text

**Course:** Natural Language Understanding & Generation  
**Dataset:** CoNLL-2003 (Reuters news wire, ~20 k sentences)  
**Goal:** Identify and classify named entities - persons, organisations, locations, and
miscellaneous - using three approaches of increasing complexity.

---

### What this notebook covers

| Part | Approach | Tool |
|------|----------|------|
| 1 | Statistical NER (pre-trained) | spaCy `en_core_web_sm` |
| 2 | Transformer NER (pre-trained) | `dbmdz/bert-large-cased-finetuned-conll03-english` |
| 3 | Fine-tuning BERT from scratch | `bert-base-cased` + HF Trainer API |
| 4 | Production deployment wrapper | Custom `NERModel` class |

---

### Entity label schemes

| spaCy label | CoNLL-2003 label | Meaning |
|-------------|-----------------|--------|
| `PERSON` | `PER` | People's names |
| `ORG` | `ORG` | Companies, agencies, institutions |
| `GPE` / `LOC` | `LOC` | Geopolitical entities and locations |
| *(various)* | `MISC` | Nationalities, events, products |

## Setup

Run the cell below once to install all dependencies into your environment.

In [ ]:
# Install all required packages (uncomment if not already installed)
# !pip install spacy transformers torch datasets seqeval scikit-learn numpy pandas accelerate

# Download the spaCy English pipeline (run once)
# !python -m spacy download en_core_web_sm

print("Environment check complete.")

---
## Part 1 - NER with spaCy

spaCy uses a **statistical CNN pipeline** trained on OntoNotes 5. It is fast, deterministic, and ships 18 entity types out of the box.  
We use `en_core_web_sm` (12 MB) for this demo.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
print(f"spaCy version : {spacy.__version__}")
print(f"Pipeline      : {nlp.pipe_names}")

### Sample sentences

Four sentences drawn from recent news - covering technology, politics, business, and international organisations.

In [ ]:
sample_texts = [
    "Tesla opened a new Gigafactory in Austin, Texas, investing over $5 billion in the facility.",
    "Narendra Modi visited Japan in May 2023 to attend the G7 summit held in Hiroshima.",
    "Amazon, led by Andy Jassy, is expanding its logistics operations across Southeast Asia.",
    "The International Monetary Fund, based in Washington D.C., revised its global growth forecast for 2024.",
]

print("=" * 72)
print(f"  {'Entity':<30} {'Label':<12} Description")
print("=" * 72)

for i, text in enumerate(sample_texts, 1):
    doc = nlp(text)
    print(f"\n[{i}] {text}")
    if doc.ents:
        for ent in doc.ents:
            print(f"     {ent.text:<30} {ent.label_:<12} {spacy.explain(ent.label_)}")
    else:
        print("     (no entities detected)")

### spaCy entity type reference

In [ ]:
entity_reference = {
    "PERSON":   "People, including fictional characters",
    "ORG":      "Companies, agencies, institutions",
    "GPE":      "Countries, cities, states (geopolitical)",
    "LOC":      "Non-GPE locations - mountains, rivers, regions",
    "DATE":     "Absolute or relative dates and periods",
    "MONEY":    "Monetary values with currency unit",
    "CARDINAL": "Numerals that do not fall under another type",
    "ORDINAL":  "Ordinal numbers - first, second, etc.",
    "MISC":     "Miscellaneous entities not covered above",
}

print(f"  {'Label':<12} Description")
print("-" * 56)
for label, desc in entity_reference.items():
    print(f"  {label:<12} {desc}")

**Observations - spaCy**
- Runs fully offline after the one-time model download.
- Recognises 18 entity types (OntoNotes scheme) vs. CoNLL-2003's 4.
- `GPE` and `LOC` are distinct: `GPE` = political entity (country, city); `LOC` = natural/geographic region.
- Typical F1 on CoNLL-2003 test: ~85 with the small model.

---
## Part 2 - NER with Hugging Face Transformers

We use **`dbmdz/bert-large-cased-finetuned-conll03-english`**, a BERT-Large model
already fine-tuned on CoNLL-2003. It achieves ~92 F1 on the test set.

`aggregation_strategy="simple"` merges consecutive sub-word tokens belonging to the
same entity into one span, so "Andy Jassy" appears as one `PER` entity rather than two tokens.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline

MODEL_NAME = "dbmdz/bert-large-cased-finetuned-conll03-english"

ner_pipeline = pipeline(
    "ner",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    aggregation_strategy="simple",
)

print(f"Model  : {MODEL_NAME}")
print(f"Device : {ner_pipeline.device}")

In [ ]:
print("=" * 72)
print(f"  {'Entity':<28} {'Label':<8} {'Score':<8} Confidence")
print("=" * 72)

for i, text in enumerate(sample_texts, 1):
    entities = ner_pipeline(text)
    print(f"\n[{i}] {text}")
    if entities:
        for ent in entities:
            bar = "#" * int(ent["score"] * 20)
            print(f"     {ent['word']:<28} {ent['entity_group']:<8} {ent['score']:.4f}  [{bar:<20}]")
    else:
        print("     (no entities detected)")

**Observations - BERT NER**
- Confidence scores are consistently above 0.99 on clear named entities.
- Sub-word splits (e.g. "D.C." → "D" + "C") can produce separate spans - a known WordPiece tokenisation artefact.
- Labels follow CoNLL-2003 scheme: `PER`, `ORG`, `LOC`, `MISC`.
- First run downloads ~1.3 GB; subsequent runs use the local HuggingFace cache.

---
## Part 3 - Fine-tuning BERT on CoNLL-2003

The complete fine-tuning pipeline consists of four steps:

1. **Load dataset** - CoNLL-2003 from the Hugging Face Hub
2. **Tokenise and align labels** - BERT's WordPiece splits words; labels must be re-aligned to sub-token boundaries
3. **Configure training** - `TrainingArguments` with warm-up learning-rate schedule
4. **Train** - using the HuggingFace `Trainer` API

> **Note:** `trainer.train()` is commented out. Uncomment it when running on a GPU-equipped machine.

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

### 3.1 - Load the CoNLL-2003 dataset

In [ ]:
# eriktks/conll2003 is a Parquet-based mirror compatible with datasets >= 3.0
DATASET_SOURCES = ["eriktks/conll2003", "conll2003"]

dataset = None
for source in DATASET_SOURCES:
    try:
        dataset = load_dataset(source)
        print(f"Loaded from   : '{source}'")
        print(f"  Train       : {len(dataset['train']):,} sentences")
        print(f"  Validation  : {len(dataset['validation']):,} sentences")
        print(f"  Test        : {len(dataset['test']):,} sentences")
        break
    except Exception as e:
        print(f"Could not load '{source}': {e}")

if dataset is None:
    print("\nFalling back to default CoNLL-2003 label set for demonstration.")

### 3.2 - Label set (IOB2 format)

In [ ]:
if dataset is not None:
    label_list = dataset["train"].features["ner_tags"].feature.names
else:
    # Standard CoNLL-2003 IOB2 label set
    label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG",
                  "B-LOC", "I-LOC", "B-MISC", "I-MISC"]

print(f"Number of labels : {len(label_list)}")
print(f"Labels           : {label_list}")
print("\nIOB2: B- = beginning of entity  |  I- = inside entity  |  O = outside")

### 3.3 - Initialise `bert-base-cased` for token classification

In [ ]:
BERT_MODEL = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
model = AutoModelForTokenClassification.from_pretrained(
    BERT_MODEL,
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)},
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Base model       : {BERT_MODEL}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

### 3.4 - Tokenise and align NER labels with sub-word tokens

BERT splits words into sub-word pieces (e.g. `Gigafactory` → `[G, ##iga, ##factory]`).  
Each **original word** has one NER label, but now maps to multiple sub-tokens.  
We assign the label to the **first sub-token** and mark continuations with `-100` (ignored by the loss).

In [ ]:
def tokenize_and_align_labels(examples, label_all_tokens=False):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    aligned_labels = []
    for i, label_seq in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)              # [CLS] / [SEP] - ignored
            elif word_idx != prev_word_idx:
                label_ids.append(label_seq[word_idx])  # first sub-token gets label
            else:
                label_ids.append(label_seq[word_idx] if label_all_tokens else -100)
            prev_word_idx = word_idx
        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized

if dataset is not None:
    tokenized_datasets = dataset.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=dataset["train"].column_names,
    )
    print("Tokenisation complete.")
    print(f"  Columns: {tokenized_datasets['train'].column_names}")
else:
    tokenized_datasets = None
    print("Tokenisation skipped - dataset not available.")

### 3.5 - Evaluation metric (entity-level F1, strict IOB2 matching)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [label_list[pred] for pred, lab in zip(row_p, row_l) if lab != -100]
        for row_p, row_l in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[lab] for pred, lab in zip(row_p, row_l) if lab != -100]
        for row_p, row_l in zip(predictions, labels)
    ]

    report = classification_report(
        true_labels, true_preds,
        mode="strict", scheme=IOB2, output_dict=True,
    )
    return {
        "precision": report["macro avg"]["precision"],
        "recall":    report["macro avg"]["recall"],
        "f1":        report["macro avg"]["f1-score"],
    }

print("compute_metrics defined - uses seqeval strict entity-level F1.")

### 3.6 - Training configuration and Trainer initialisation

In [ ]:
training_args = TrainingArguments(
    output_dir                  = "./bert-ner-finetuned",
    eval_strategy               = "epoch",      # evaluate after every epoch
    save_strategy               = "epoch",
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,          # 10% of steps for LR warm-up
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 3,
    weight_decay                = 0.01,
    logging_steps               = 50,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    push_to_hub                 = False,
)

if tokenized_datasets is not None:
    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = tokenized_datasets["train"],
        eval_dataset    = tokenized_datasets["validation"],
        tokenizer       = tokenizer,
        data_collator   = data_collator,
        compute_metrics = compute_metrics,
    )
    print("Trainer initialised and ready.")
    print("Uncomment trainer.train() below to begin training.")
else:
    trainer = None
    print("Trainer not initialised - dataset not available.")

In [ ]:
# ── Training (uncomment on a GPU-equipped machine) ───────────────────────
# trainer.train()

# ── Evaluate on held-out test set (after training) ───────────────────────
# test_results = trainer.evaluate(tokenized_datasets['test'])
# print(f'Test results: {test_results}')

# ── Save the fine-tuned model ─────────────────────────────────────────────
# trainer.save_model('./my-ner-model')
# tokenizer.save_pretrained('./my-ner-model')
# print('Model saved to ./my-ner-model')

print("Training cell ready. Uncomment the lines above to execute on a GPU machine.")

**Observations - Fine-tuning**
- `warmup_ratio=0.1` ramps the learning rate from 0 over the first 10% of training steps, stabilising early gradient updates.
- `eval_strategy="epoch"` checkpoints and selects the best model by F1 after each epoch.
- Sub-word label alignment with `-100` masking ensures only one label contributes to the loss per word.
- Expected training time: ~25 min on a single T4 GPU (3 epochs, 14 k sentences, batch 16).

---
## Part 4 - Model Deployment

We package the inference logic in a reusable `NERModel` class with two improvements:

1. **Confidence threshold** (`min_score=0.80`) - filters out low-confidence predictions
2. **Native batch processing** - uses the HuggingFace pipeline's `batch_size` argument, significantly faster than a Python loop

In [ ]:
import torch
from transformers import pipeline as hf_pipeline


class NERModel:
    # Production-ready NER inference wrapper

    def __init__(self, model_path='dbmdz/bert-large-cased-finetuned-conll03-english', min_score=0.80):
        self.min_score = min_score
        self.pipeline = hf_pipeline(
            'ner',
            model=model_path,
            tokenizer=model_path,
            aggregation_strategy='simple',
            device=0 if torch.cuda.is_available() else -1,
        )

    def extract_entities(self, text):
        # Return entities above the confidence threshold
        return [
            {
                'text':  ent['word'],
                'label': ent['entity_group'],
                'score': round(float(ent['score']), 4),
                'start': ent['start'],
                'end':   ent['end'],
            }
            for ent in self.pipeline(text)
            if ent['score'] >= self.min_score
        ]

    def extract_entities_batch(self, texts, batch_size=8):
        # Extract entities from multiple texts using native pipeline batching
        raw_results = self.pipeline(texts, batch_size=batch_size)
        return [
            [
                {
                    'text':  ent['word'],
                    'label': ent['entity_group'],
                    'score': round(float(ent['score']), 4),
                }
                for ent in result
                if ent['score'] >= self.min_score
            ]
            for result in raw_results
        ]


ner_model = NERModel(min_score=0.80)
print('NERModel initialised.')
print(f'  Model     : dbmdz/bert-large-cased-finetuned-conll03-english')
print(f'  Device    : {ner_model.pipeline.device}')
print(f'  Min score : {ner_model.min_score}')

### 4.1 - Single-text inference

In [ ]:
test_text = "Tesla CEO Elon Musk announced a new Gigafactory in Austin, Texas."

entities = ner_model.extract_entities(test_text)

print(f"Input : {test_text}")
print(f"\n  {'Entity':<28} {'Label':<8} {'Score':<8} Span")
print("  " + "-" * 60)
for e in entities:
    print(f"  {e['text']:<28} {e['label']:<8} {e['score']:<8} [{e['start']}:{e['end']}]")

### 4.2 - Batch inference

In [ ]:
news_articles = [
    "Reliance Industries reported record profits of $10 billion in fiscal year 2023.",
    "Sundar Pichai led Google to launch Gemini AI in December 2023 from its headquarters in Mountain View.",
    "The World Trade Organization held its annual conference in Geneva in February 2024.",
]

batch_results = ner_model.extract_entities_batch(news_articles)

for i, (text, ents) in enumerate(zip(news_articles, batch_results), 1):
    print(f"\n[{i}] {text}")
    if ents:
        for e in ents:
            print(f"     {e['text']:<32} {e['label']:<8} score={e['score']:.4f}")
    else:
        print("     (no entities above threshold)")

---
## Summary

| Approach | Model size | F1 (CoNLL-2003 test) | Key trade-off |
|----------|-----------|---------------------|---------------|
| spaCy `en_core_web_sm` | 12 MB | ~85 | Fast, offline, 18 label types |
| BERT-Large fine-tuned | 1.3 GB | ~92 | Highest accuracy, GPU recommended |
| BERT-Base fine-tuned | 420 MB | ~90 | Good balance of size and accuracy |

### What was demonstrated

1. **spaCy NER** - lightweight statistical pipeline; 18 entity types; ideal when latency and offline use matter
2. **Pre-trained BERT NER** - state-of-the-art accuracy with zero additional training
3. **Fine-tuning BERT** - full training loop with sub-word label alignment, warm-up scheduling, and entity-level F1 evaluation
4. **Deployment wrapper** - `NERModel` class with confidence filtering and native batching

### Suggested next steps
- Run `trainer.train()` on a GPU to produce a custom fine-tuned checkpoint
- Serve `NERModel` via a **FastAPI** endpoint for real-time REST inference
- Extend the label set for domain-specific entities (e.g. drug names, financial instruments)